In [3]:
from numba import jit
from numba import prange
import numpy as np
import quantecon as qe


In [5]:
@jit
def montecarlo_pi(n):
    count = 0
    for i in range(n):
        x, y = np.random.uniform(), np.random.uniform()
        d = np.sqrt((x - 0.5)**2 + (y - 0.5)**2)    
        if d < 0.5:
            count += 1
    area_aprox = count / n
    return(area_aprox * 4)

with qe.Timer():
    montecarlo_pi(1000000)


0.0613 seconds elapsed


In [6]:
def dailyvol(n):
    count = 0
    previous_state = 1
    sequence = np.empty(n, dtype=np.int64)
    for i in range(n):
        x = np.random.uniform() 
        current_state = previous_state
        if previous_state == 1:
            if x > 0.8:
                current_state = 0
        if previous_state == 0:
            if x > 0.9:
                current_state = 1
        sequence[i] = current_state
        previous_state = current_state
        count += current_state       
    return (sequence, count)

n = 1000000
print(dailyvol(n))

with qe.Timer():
    dailyvol(n)

dailyvol_numba = jit(dailyvol)
print(dailyvol_numba(n))

with qe.Timer():
    dailyvol_numba(n)

#For numba code, this direct simple loop is best. But using numpy and generating all my variables prior to the loop would be better




(array([1, 1, 1, ..., 1, 0, 0], shape=(1000000,)), 332536)
0.8603 seconds elapsed
(array([0, 0, 0, ..., 1, 1, 0], shape=(1000000,)), 335416)
0.0063 seconds elapsed


In [8]:
@jit(parallel=True)
def montecarlo_pi(n):
    count = 0
    for i in prange(n):
        x, y = np.random.uniform(), np.random.uniform()
        d = np.sqrt((x - 0.5)**2 + (y - 0.5)**2)    
        if d < 0.5:
            count += 1
    area_aprox = count / n
    return(area_aprox * 4)

with qe.Timer():
    montecarlo_pi(100000000)


0.2045 seconds elapsed


In [10]:
@jit(parallel=True)
def stoch_vol_call(M=10000000):
    n, β, K = 20, 0.99, 100.
    mu, rho, nu, = 0.0001, 0.1, 0.001
    total_payoff = 0
    for m in prange(M):
        s = 10
        h = 0
        for t in range(n):
            xi, eta = np.random.standard_normal(), np.random.standard_normal()
            vol = np.exp(h)
            s = s * np.exp(mu + vol * xi)
            h = rho * h + nu * eta

        total_payoff += max(s - K, 0)
    p = (β**n) * (total_payoff/M)
    return p


stoch_vol_call()




120760.90824202911